# Pré-natal, raça/cor e escolaridade

Este notebook explora como escolaridade materna e realização de pré-natal se distribuem entre casos de sífilis congênita por grupo racial em Porto Alegre.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

import pandas as pd
from src.etl.dbc import read_dbc

CODIGO_PORTO_ALEGRE = "431490"

In [ ]:
def grupo_racial(valor):
    codigo = str(valor).strip()
    if codigo in {"2", "4"}:
        return "Mães negras"
    if codigo in {"1", "3", "5"}:
        return "Mães não negras"
    return "Ignorado/sem informação"


def escolaridade(valor):
    codigo = str(valor).strip()
    if codigo in {"02", "03", "04", "05"}:
        return "Até 7 anos de estudo"
    if codigo in {"06", "07", "08"}:
        return "8 anos ou mais de estudo"
    return "Ignorada/sem informação"


def pre_natal(valor):
    codigo = str(valor).strip()
    if codigo == "1":
        return "Com pré-natal"
    if codigo == "2":
        return "Sem pré-natal"
    return "Ignorado/sem informação"

In [ ]:
sinan = read_dbc(ROOT / "data/raw/SIFCBR24.dbc")
casos = sinan[sinan["ID_MN_RESI"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)].copy()

casos["grupo_racial"] = casos["ANT_RACA"].map(grupo_racial)
casos["escolaridade_mae"] = casos["ESCOLMAE"].map(escolaridade)
casos["pre_natal"] = casos["ANT_PRE_NA"].map(pre_natal)

casos[["grupo_racial", "pre_natal", "escolaridade_mae"]].value_counts().reset_index(name="casos")

In [ ]:
sem_prenatal = casos[casos["pre_natal"].eq("Sem pré-natal")].copy()
resumo = (
    sem_prenatal
    .groupby(["grupo_racial", "escolaridade_mae"], as_index=False)
    .size()
    .rename(columns={"size": "casos"})
)
resumo["total_grupo"] = resumo.groupby("grupo_racial")["casos"].transform("sum")
resumo["percentual_no_grupo"] = (resumo["casos"] / resumo["total_grupo"] * 100).round(1)
resumo.sort_values(["grupo_racial", "escolaridade_mae"])